# MGE-LDM Total Generation

Run these cells from top to bottom in Colab. The notebook installs MGE-LDM, downloads the required checkpoints, generates audio, organizes the WAV files, and downloads a ZIP bundle. It does not run any evaluation code.

## 1. Install system tools and clone MGE-LDM

In [ ]:
from pathlib import Path
import os
import subprocess

ROOT = Path("/content")
REPO = ROOT / "MGE-LDM"
MAMBA = ROOT / "bin" / "micromamba"

os.environ["MAMBA_ROOT_PREFIX"] = str(ROOT / "micromamba")

def run(cmd, cwd=None):
    print("\n$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)

run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "ffmpeg", "libsndfile1", "git", "curl", "bzip2"])

if not MAMBA.exists():
    run([
        "bash", "-lc",
        "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba",
    ])

if not REPO.exists():
    run(["git", "clone", "https://github.com/yoongi43/MGE-LDM.git"], cwd=ROOT)

print("Micromamba:", MAMBA.exists(), MAMBA)
print("Repo:", REPO.exists(), REPO)

## 2. Create the Python environment

Set `REBUILD_ENV = True` if the environment exists but package installation failed earlier.

In [ ]:
from pathlib import Path
import os
import subprocess

ROOT = Path("/content")
REPO = ROOT / "MGE-LDM"
MAMBA = ROOT / "bin" / "micromamba"

ENV_NAME = "mgeldm"
ENV_PREFIX = ROOT / "micromamba" / "envs" / ENV_NAME
ENV_LIB = ENV_PREFIX / "lib"
REBUILD_ENV = False

os.environ["MAMBA_ROOT_PREFIX"] = str(ROOT / "micromamba")

def mge_env():
    env = os.environ.copy()
    env["MAMBA_ROOT_PREFIX"] = str(ROOT / "micromamba")
    env["MAX_JOBS"] = "4"

    if ENV_PREFIX.exists():
        env["CUDA_HOME"] = str(ENV_PREFIX)
        env["PATH"] = f"{ENV_PREFIX / 'bin'}:{env.get('PATH', '')}"
        env["LD_LIBRARY_PATH"] = f"{ENV_LIB}:{env.get('LD_LIBRARY_PATH', '')}"

    return env

def run_live(cmd, cwd=None):
    print("\n$", " ".join(map(str, cmd)))
    process = subprocess.Popen(
        list(map(str, cmd)),
        cwd=cwd,
        env=mge_env(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    for line in process.stdout:
        print(line, end="")
        lines.append(line)

    code = process.wait()
    if code != 0:
        print("\nLAST OUTPUT:")
        print("".join(lines[-120:]))
        raise RuntimeError(f"Command failed with code {code}")

if ENV_PREFIX.exists() and REBUILD_ENV:
    print("Removing old MGE-LDM environment...")
    run_live([MAMBA, "env", "remove", "-y", "-n", ENV_NAME])

if not ENV_PREFIX.exists():
    run_live([
        MAMBA, "create", "-y", "-n", ENV_NAME,
        "-c", "nvidia",
        "-c", "conda-forge",
        "python=3.9",
        "pip=24.3.1",
        "setuptools=69.5.1",
        "wheel",
        "packaging",
        "ninja",
        "numpy=1.26.4",
        "ffmpeg",
        "libsndfile",
        "libstdcxx-ng",
        "libgcc-ng",
        "cuda-nvcc=12.1",
        "cuda-cudart-dev=12.1",
    ])

    run_live([
        MAMBA, "run", "-n", ENV_NAME,
        "python", "-m", "pip", "install",
        "--no-cache-dir",
        "--index-url", "https://download.pytorch.org/whl/cu121",
        "torch==2.3.1",
        "torchvision==0.18.1",
        "torchaudio==2.3.1",
    ])

    print("\nChecking CUDA compiler...")
    run_live([
        MAMBA, "run", "-n", ENV_NAME,
        "bash", "-lc",
        "which nvcc && nvcc --version",
    ])

    print("\nInstalling flash-attn...")
    run_live([
        MAMBA, "run", "-n", ENV_NAME,
        "python", "-m", "pip", "install",
        "--no-cache-dir",
        "--no-build-isolation",
        "flash-attn==2.7.0.post2",
    ])

    req_path = REPO / "requirements.txt"
    filtered_req = ROOT / "mge_requirements_without_torch_flash.txt"
    skip_tokens = ["flash-attn", "flash_attn", "torch==", "torchvision==", "torchaudio=="]

    filtered_lines = []
    skipped_lines = []
    for line in req_path.read_text().splitlines():
        if any(token in line.strip() for token in skip_tokens):
            skipped_lines.append(line)
        else:
            filtered_lines.append(line)

    filtered_req.write_text("\n".join(filtered_lines) + "\n")

    print("\nSkipped from requirements.txt:")
    for line in skipped_lines:
        print(" ", line)

    print("\nInstalling remaining MGE-LDM requirements...")
    run_live([
        MAMBA, "run", "-n", ENV_NAME,
        "python", "-m", "pip", "install",
        "--no-cache-dir",
        "-r", filtered_req,
    ])

    print("\nInstalling notebook helper packages...")
    run_live([
        MAMBA, "run", "-n", ENV_NAME,
        "python", "-m", "pip", "install",
        "--no-cache-dir",
        "gdown",
        "soundfile",
        "librosa",
        "tqdm",
        "einops",
        "omegaconf",
        "hydra-core",
        "safetensors",
        "accelerate",
        "alias-free-torch==0.0.6",
    ])
else:
    print("Using existing MGE-LDM environment:", ENV_PREFIX)

print("\nFinal environment check...")
run_live([
    MAMBA, "run", "-n", ENV_NAME,
    "python", "-c",
    """
import torch
import torchaudio
import numpy
import soundfile
import librosa
import flash_attn
import alias_free_torch

print("torch:", torch.__version__)
print("torchaudio:", torchaudio.__version__)
print("numpy:", numpy.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)
print("flash_attn:", flash_attn.__file__)
print("alias_free_torch:", alias_free_torch.__file__)
print("MGE-LDM env ready.")
""",
])

## 3. Download checkpoints

In [ ]:
from pathlib import Path
import os
import subprocess

ROOT = Path("/content")
REPO = ROOT / "MGE-LDM"
MAMBA = ROOT / "bin" / "micromamba"
ENV_NAME = "mgeldm"

CKPT_DIR = REPO / "ckpts"
CLAP_DIR = CKPT_DIR / "clap"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
CLAP_DIR.mkdir(parents=True, exist_ok=True)

os.environ["MAMBA_ROOT_PREFIX"] = str(ROOT / "micromamba")

dit_ckpt = CKPT_DIR / "unwrapped_DiT.ckpt"
clap_ckpt = CLAP_DIR / "music_audioset_epoch_15_esc_90.14.pt"
expected_clap_path = Path("/data2/yoongi/dataset/pre_trained/music_audioset_epoch_15_esc_90.14.pt")

def run(cmd):
    print("\n$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)

def download_if_missing(url, out_path, min_size):
    if out_path.exists() and out_path.stat().st_size > min_size:
        print("Already downloaded:", out_path)
        return

    if out_path.exists():
        out_path.unlink()

    run([
        MAMBA, "run", "-n", ENV_NAME,
        "python", "-m", "gdown",
        "--fuzzy",
        url,
        "-O",
        out_path,
    ])

download_if_missing(
    "https://drive.google.com/file/d/1tyND8iI5Whs6_Oe-pBK2SpysGLKKa6sR/view?usp=sharing",
    dit_ckpt,
    100_000_000,
)

if not clap_ckpt.exists() or clap_ckpt.stat().st_size < 1_000_000_000:
    run([
        MAMBA, "run", "-n", ENV_NAME,
        "python", "-m", "pip", "install",
        "--no-cache-dir",
        "huggingface_hub",
        "hf_xet",
    ])

    run([
        MAMBA, "run", "-n", ENV_NAME,
        "python", "-c",
        f"""
from pathlib import Path
from huggingface_hub import hf_hub_download

out_dir = Path("{CLAP_DIR}")
out_dir.mkdir(parents=True, exist_ok=True)

path = hf_hub_download(
    repo_id="lukewys/laion_clap",
    filename="music_audioset_epoch_15_esc_90.14.pt",
    local_dir=str(out_dir),
    local_dir_use_symlinks=False,
)

print("Downloaded CLAP checkpoint:", path)
""",
    ])
else:
    print("CLAP checkpoint already downloaded:", clap_ckpt)

expected_clap_path.parent.mkdir(parents=True, exist_ok=True)
if expected_clap_path.exists() or expected_clap_path.is_symlink():
    expected_clap_path.unlink()
expected_clap_path.symlink_to(clap_ckpt)

print("\nDiT checkpoint:")
print("path:", dit_ckpt)
print("exists:", dit_ckpt.exists())
print("size GB:", round(dit_ckpt.stat().st_size / 1024**3, 2) if dit_ckpt.exists() else None)

print("\nCLAP checkpoint:")
print("path:", clap_ckpt)
print("exists:", clap_ckpt.exists())
print("size GB:", round(clap_ckpt.stat().st_size / 1024**3, 2) if clap_ckpt.exists() else None)

print("\nHardcoded CLAP symlink:")
print("path:", expected_clap_path)
print("exists:", expected_clap_path.exists())
print("resolves to:", expected_clap_path.resolve() if expected_clap_path.exists() else None)

if not dit_ckpt.exists():
    raise RuntimeError("Missing MGE-LDM DiT checkpoint.")
if not clap_ckpt.exists():
    raise RuntimeError("Missing CLAP checkpoint.")
if not expected_clap_path.exists():
    raise RuntimeError("Hardcoded CLAP symlink was not created correctly.")

## 4. Choose generation settings

In [ ]:
from pathlib import Path

ROOT = Path("/content")
REPO = ROOT / "MGE-LDM"

N_GENERATIONS = 151
GEN_AUDIO_DUR = 30.0
NUM_STEPS = 50
CFG_SCALE = 6.0
OVERLAP_DUR = 5.0
REPAINT_N = 4

TEXT_PROMPT = "Instrumental music with bass, drums, guitar, and piano"
PROMPTS = [TEXT_PROMPT for _ in range(N_GENERATIONS)]

output_root = REPO / "outputs_total_generation" / f"mge_total_gen_{N_GENERATIONS}samples_{NUM_STEPS}steps"
output_root.mkdir(parents=True, exist_ok=True)

print("Output folder:", output_root)
print("Prompt:", TEXT_PROMPT)

## 5. Generate audio

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import time

ROOT = Path("/content")
REPO = ROOT / "MGE-LDM"
MAMBA = ROOT / "bin" / "micromamba"
ENV_NAME = "mgeldm"

dit_ckpt = REPO / "ckpts" / "unwrapped_DiT.ckpt"
output_root.mkdir(parents=True, exist_ok=True)

config = {
    "model": "MGE-LDM",
    "task": "total_gen",
    "n_generations": N_GENERATIONS,
    "gen_audio_dur": GEN_AUDIO_DUR,
    "num_steps": NUM_STEPS,
    "cfg_scale": CFG_SCALE,
    "overlap_dur": OVERLAP_DUR,
    "repaint_n": REPAINT_N,
    "checkpoint": str(dit_ckpt),
    "prompts": PROMPTS,
}

with open(output_root / "generation_config.json", "w") as f:
    json.dump(config, f, indent=2)

def run_live(cmd, cwd=None, env=None):
    print("\n$", " ".join(map(str, cmd)))
    process = subprocess.Popen(
        list(map(str, cmd)),
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    for line in process.stdout:
        print(line, end="")
        lines.append(line)

    code = process.wait()
    if code != 0:
        print("\nLAST OUTPUT:")
        print("".join(lines[-120:]))
        raise RuntimeError(f"Command failed with code {code}")

env = os.environ.copy()
env["MAMBA_ROOT_PREFIX"] = str(ROOT / "micromamba")
env["PYTHONPATH"] = str(REPO)
env["HYDRA_FULL_ERROR"] = "1"

done_marker = output_root / "generation_done.txt"

if done_marker.exists():
    print("Generation already marked done. Skipping.")
else:
    for i, prompt in enumerate(PROMPTS):
        sample_out_dir = output_root / "raw_infer_outputs"

        print(f"\n=== Generating sample {i + 1}/{N_GENERATIONS} ===")
        print("Prompt:", prompt)
        print("Time:", time.strftime("%Y-%m-%d %H:%M:%S"))

        cmd = [
            str(MAMBA), "run", "-n", ENV_NAME,
            "python", "-u", "infer.py",
            "--config-name", "dit",
            "+task=total_gen",
            f"ckpt_path={dit_ckpt}",
            f"+gen_audio_dur={GEN_AUDIO_DUR}",
            "+given_wav_path=null",
            f"+text_prompt='{prompt}'",
            f"+num_steps={NUM_STEPS}",
            f"+cfg_scale={CFG_SCALE}",
            f"+overlap_dur={OVERLAP_DUR}",
            f"+repaint_n={REPAINT_N}",
            f"+output_dir={sample_out_dir}",
        ]

        run_live(cmd, cwd=REPO, env=env)

    done_marker.write_text("done\n")

print("Raw generation output:", output_root / "raw_infer_outputs")

## 6. Organize generated WAV files

In [ ]:
from pathlib import Path
import json
import shutil

raw_dir = output_root / "raw_infer_outputs" / "total_gen"
clean_dir = output_root / "clean"

gen_mix_dir = clean_dir / "gen_mix"
gen_submix_dir = clean_dir / "gen_submix"
gen_src_dir = clean_dir / "gen_src"
gen_tracks_dir = clean_dir / "gen_tracks"

for path in [gen_mix_dir, gen_submix_dir, gen_src_dir, gen_tracks_dir]:
    path.mkdir(parents=True, exist_ok=True)

print("raw_dir:", raw_dir, raw_dir.exists())
if not raw_dir.exists():
    raise RuntimeError("Raw total-generation output folder is missing.")

output_dirs = sorted(raw_dir.glob("output_*"))
print("Found output dirs:", len(output_dirs))
if not output_dirs:
    raise RuntimeError("No output_* folders found.")

manifest = []

for idx, out_dir in enumerate(output_dirs):
    sample_name = f"Sample{idx:05d}"
    sample_dir = gen_tracks_dir / sample_name
    sample_dir.mkdir(parents=True, exist_ok=True)

    files = {
        "gen_mix": out_dir / "gen_mix.wav",
        "gen_submix": out_dir / "gen_submix.wav",
        "gen_src": out_dir / "gen_src.wav",
    }

    for label, src in files.items():
        if not src.exists():
            print("Missing:", src)
            continue

        if label == "gen_mix":
            shutil.copy2(src, gen_mix_dir / f"{sample_name}.wav")
        elif label == "gen_submix":
            shutil.copy2(src, gen_submix_dir / f"{sample_name}.wav")
            shutil.copy2(src, sample_dir / "submix.wav")
        elif label == "gen_src":
            shutil.copy2(src, gen_src_dir / f"{sample_name}.wav")
            shutil.copy2(src, sample_dir / "src.wav")

    prompt_file = out_dir / "prompt.txt"
    if prompt_file.exists():
        shutil.copy2(prompt_file, sample_dir / "prompt.txt")

    manifest.append({
        "sample": sample_name,
        "raw_output_dir": str(out_dir),
        "has_mix": (gen_mix_dir / f"{sample_name}.wav").exists(),
        "has_submix": (sample_dir / "submix.wav").exists(),
        "has_src": (sample_dir / "src.wav").exists(),
    })

with open(clean_dir / "sample_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Clean generation folder:", clean_dir)
print("gen_mix files:", len(list(gen_mix_dir.glob("*.wav"))))
print("gen_submix files:", len(list(gen_submix_dir.glob("*.wav"))))
print("gen_src files:", len(list(gen_src_dir.glob("*.wav"))))
print("gen_tracks:", len(list(gen_tracks_dir.glob("Sample*"))))

## 7. Download generated audio

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

zip_path = Path("/content/mge_ldm_total_generation_samples.zip")
clean_dir = output_root / "clean"

if zip_path.exists():
    zip_path.unlink()

files_added = 0
audio_folders = ["gen_mix", "gen_submix", "gen_src", "gen_tracks"]
metadata_files = [
    output_root / "generation_config.json",
    clean_dir / "sample_manifest.json",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as zf:
    for folder_name in audio_folders:
        folder = clean_dir / folder_name
        if not folder.exists():
            print("Skipped missing folder:", folder)
            continue

        for path in sorted(folder.rglob("*")):
            if path.is_file():
                arcname = Path("samples") / folder_name / path.relative_to(folder)
                zf.write(path, arcname)
                files_added += 1

    for path in metadata_files:
        if path.exists():
            zf.write(path, Path("metadata") / path.name)
            files_added += 1

if files_added == 0:
    raise RuntimeError("No generated audio was added to the ZIP.")

print("ZIP created:", zip_path)
print("Files added:", files_added)
print("ZIP size GB:", round(zip_path.stat().st_size / 1024**3, 2))

files.download(str(zip_path))